# Kernel cuantico para QSVM

Construye el kernel de fidelidad ``K(x_i, x_j) = P(00...0)`` del circuito
``U(x_j)^dagger U(x_i)``, con el feature map elegido en `FEATURE_MAP`
(ZZ, Pauli Z+YY o Ry-CX-Rx) definido en pytket y ejecutado via guppy.

Flujo: cargar datos -> inspeccionar circuitos (sin shots) -> construir la matriz
kernel. Toda la logica vive en `funciones_nexus.py`; aqui solo quedan los
parametros y las llamadas.

La construccion de la matriz esta apagada por defecto (`RUN_MATRIX = False`):
revisa circuitos y costo antes de encenderla.

## 1. Configuracion y datos

Carga el dataset escalado y lo separa en train/test segun `_PartInd_`.

In [ ]:
import pandas as pd
from IPython.display import display
from pytket.circuit.display import render_circuit_jupyter

# Los valores del kernel suelen ser pequenos; se muestran con 6 decimales
# fijos (sin notacion cientifica) para poder distinguirlos.
pd.set_option("display.float_format", "{:.6f}".format)

from funciones_nexus import (
    cargar_datos_kernel,
    obtener_feature_map,
    seleccionar_par_kernel,
    iniciar_matriz_kernel,
    iniciar_matriz_kernel_test,
    consultar_matriz_nexus,
    guardar_kernel_qsvm,
    FEATURE_MAPS,
    MATRIX_BACKEND_OPTIONS,
)

# Feature map del kernel: elige una de las 3 opciones de FEATURE_MAPS.
#   "zz"       -> ZZFeatureMap (fases Z + interacciones ZZ)
#   "zyy"      -> Pauli Z+YY explicito (entrelazamiento lineal)
#   "ry_cx_rx" -> Ry -> cadena CX -> Rx
# El mismo FEATURE_MAP gobierna K_train y K_test (deben coincidir).
FEATURE_MAP = "zz"

PROJECT_NAME = "prueba_migracion"                       # Proyecto de Nexus (se crea si no existe)
KERNEL_DATA_PATH = "data/processed/df_escalado.csv"     # Dataset escalado con _PartInd_

kernel_df, kernel_feature_columns, kernel_train_df, kernel_test_df = cargar_datos_kernel(KERNEL_DATA_PATH)
print("Feature map:", FEATURE_MAP, "| opciones:", list(FEATURE_MAPS))
print("Features:", kernel_feature_columns)
print(f"Train: {kernel_train_df.shape} | Test: {kernel_test_df.shape}")
display(kernel_train_df.head())

## 2. Inspeccion del feature map U(x)

No consume shots.

In [ ]:
PREVIEW_ROW = 0     # Cambia esta fila para inspeccionar otro U(x), sin ejecutar shots

preview_x = kernel_train_df.iloc[PREVIEW_ROW].to_numpy(dtype=float)
feature_map_fn = obtener_feature_map(FEATURE_MAP)
feature_map_preview = feature_map_fn(preview_x)
print(f"Feature map '{FEATURE_MAP}' de train[{PREVIEW_ROW}] | qubits: {feature_map_preview.n_qubits} | puertas: {feature_map_preview.n_gates}")
render_circuit_jupyter(feature_map_preview)

## 3. Seleccion e inspeccion del par

Construye ``U(x_j)^dagger U(x_i)`` con barreras para revisarlo antes de ejecutar.

In [ ]:
KERNEL_ROW_I = 0    # Filas de train que forman el par
KERNEL_ROW_J = 1

kernel_x_i, kernel_x_j, kernel_preview_circuit = seleccionar_par_kernel(
    kernel_train_df, KERNEL_ROW_I, KERNEL_ROW_J, feature_map=FEATURE_MAP
)
render_circuit_jupyter(kernel_preview_circuit)

## 4. Matriz kernel (train o test)

El interruptor `MATRIX_KIND` decide que matriz construye esta seccion:

- `"train"` -> `K_train = K(X_train, X_train)`, cuadrada, para `SVC.fit`.
- `"test"`  -> `K_test = K(X_test, X_train)`, rectangular, para `SVC.predict`.

Ambas usan el mismo `FEATURE_MAP`. Se ejecuta el triangulo superior y se
refleja por simetria; para `m` filas de train K_train requiere `m(m-1)/2`
circuitos. K_test apila `[test, train]`, construye la conjunta y recorta el
bloque test x train (computa de mas los bloques test-test y train-train).

Backends: Selene local o Nexus (Selene, H1/H2 via compile job de pytket, Helios).
Tras enviar a Nexus, **no reejecutes esta celda**: usa la celda de consulta.

In [ ]:
MATRIX_KIND = "train"                        # "train" -> K(train,train) ; "test" -> K(test,train)
MATRIX_ROWS = [0, 1, 2, 3]                   # Filas de train (columnas del kernel en ambos casos)
TEST_ROWS = [0, 1, 2, 3]                     # Filas de test (solo para MATRIX_KIND="test")
MATRIX_BACKEND = "H2-EMULATOR"               # Ver MATRIX_BACKEND_OPTIONS
RUN_MATRIX = True                            # Interruptor de seguridad
MATRIX_SHOTS = 10
MATRIX_SEED = 42
MATRIX_EXECUTE_DIAGONAL = True               # Solo K_train; False fija K(i,i)=1 sin ejecutar
SAVE_MATRIX_RUN = True

if MATRIX_KIND == "train":
    matrix_state, matrix_result = iniciar_matriz_kernel(
        kernel_train_df, MATRIX_ROWS, MATRIX_BACKEND, RUN_MATRIX,
        n_shots=MATRIX_SHOTS, seed=MATRIX_SEED,
        ejecutar_diagonal=MATRIX_EXECUTE_DIAGONAL,
        guardar=SAVE_MATRIX_RUN, project_name=PROJECT_NAME,
        feature_map=FEATURE_MAP,
    )
    row_labels, col_labels = MATRIX_ROWS, MATRIX_ROWS
elif MATRIX_KIND == "test":
    matrix_state, matrix_result = iniciar_matriz_kernel_test(
        kernel_train_df, MATRIX_ROWS, kernel_test_df, MATRIX_BACKEND, RUN_MATRIX,
        test_rows=TEST_ROWS, n_shots=MATRIX_SHOTS, seed=MATRIX_SEED,
        guardar=SAVE_MATRIX_RUN, project_name=PROJECT_NAME,
        feature_map=FEATURE_MAP,
    )
    row_labels, col_labels = TEST_ROWS, MATRIX_ROWS
else:
    raise ValueError("MATRIX_KIND debe ser 'train' o 'test'.")

if matrix_result is not None:
    display(pd.DataFrame(matrix_result["kernel_matrix"], index=row_labels, columns=col_labels))
    display(matrix_result["run_summary"])

## 5. Consulta de la matriz remota

Reejecutar solo esta celda. Para H1/H2 encadena compile -> execute automaticamente;
al completarse reconstruye la matriz y guarda el CSV.

In [ ]:
matrix_state, matrix_result_remoto = consultar_matriz_nexus(matrix_state, guardar=SAVE_MATRIX_RUN)

if matrix_result_remoto is not None:
    remote_rows = matrix_result_remoto["row_labels"]
    remote_cols = matrix_result_remoto.get("col_labels", remote_rows)
    display(pd.DataFrame(matrix_result_remoto["kernel_matrix"], index=remote_rows, columns=remote_cols))
    display(matrix_result_remoto["run_summary"])

## 6. Guardado del kernel para QSVM

Persiste la matriz de Gram y un CSV de metadatos con su procedencia
(backend, job_id, shots, filas, timestamp). Segun `MATRIX_KIND` guarda
`K_train` (cuadrada, `kernel_qsvm_<run_id>.csv`) o `K_test` (rectangular,
`kernel_qsvm_test_<run_id>.csv`). Toma la matriz local o la remota, la que
este disponible.

In [ ]:
# Guarda la matriz de Gram + metadatos (data/runs/kernel_qsvm[_test]_*.csv).
if matrix_result_remoto is not None:
    resultado_final = matrix_result_remoto
    fuente = f"nexus_{MATRIX_BACKEND}_{MATRIX_KIND}"
    id_job = matrix_state["job_ref"].id
elif matrix_result is not None:
    resultado_final = matrix_result
    fuente = f"local_statevector_{MATRIX_KIND}"
    id_job = None
else:
    resultado_final = None
    print("Aun no hay una matriz kernel construida que guardar. Corre el paso 4 (o 5).")

if resultado_final is not None:
    ruta_kernel, ruta_meta = guardar_kernel_qsvm(resultado_final, source=fuente, job_id=id_job)
    print("Matriz kernel guardada en:", ruta_kernel)
    print("Metadatos en:", ruta_meta)
    display(pd.read_csv(ruta_kernel, sep=";", index_col=0))